# Microdroplet Screen Controller — Tutorial

Quick-start guide to the microdroplet controller app.

This application is primarily meant to help you troubleshoot the very first installation process, and any other common errors that may pop up along the way

A recorded walkthrough is also in `microdroplet app demo.mp4`.

## 1. Install & launch

- Python **3.10+** (tested on 3.12)
- Copy the whole `microdroplet_app` folder to your machine, keeping all files together
- Install dependencies from `requirements.txt` (`pandas`, `pyserial`, `matplotlib` — matplotlib only powers the optional plate-visualization feature)

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sys, os

APP_DIR = os.getcwd()  

In [4]:
import subprocess

subprocess.Popen([sys.executable, 'main.py'], cwd=APP_DIR)
print("Launched — check your taskbar for the app window (or just run 'python main.py' yourself).")

Launched — check your taskbar for the app window (or just run 'python main.py' yourself).


## 2. Configure settings

**Configure tab → left panel**, all auto-saved to `microdroplet_config.json` as you change them:
- **PARAMETERS** — PFD/organic dwell time, pause after pump stop, droplet multiplicity (repeats), feedrate
- **MACHINE SETTINGS ($110–$122)** — max rate & acceleration per axis; keep these matched to your CNC controller's real GRBL settings so the time estimate (§6) is accurate. `Reset to Defaults` restores the stock values.
- **OUTPUT LOCATIONS** — G-code folder & Combinations CSV folder, each set once via `Browse…`; reused automatically every time you generate G-code.

<img src="tutorial_screenshots/settings_panel.png" width="420">

## 3. Build or load a well plate

**Configure tab → right panel (Plate Editor):**
- **Size** — pick 6 / 12 / 24 / 48 / 96 / 384 wells, or `Custom` rows/cols/spacing/origin
- Click a well to set its **chemical name** and **class**. One well needs class `pfd`/`perfluorodecalin`/`oil` (the carrier fluid). Every other *unique* class becomes one dimension of the combinatorial grid — e.g. 3 `amine` wells × 2 `aldehyde` wells → 6 pairings — so give each reagent role its own class name.
- **GEOMETRY PRESETS** save/load a rows/cols/spacing/origin layout by name; **REACTION PRESETS** save/load well contents by name — both live inside `microdroplet_config.json`. **CSV Import/Export** reads/writes a standalone, portable `.csv` (needs columns `plate_row`, `plate_column`, `chemical_name`, `class`).

<img src="tutorial_screenshots/plate_editor.png" width="700">

import pandas as pd

sys.path.insert(0, APP_DIR)  # set up in section 1
from gcode_generator import Screen

demo_df = pd.DataFrame([
    {'plate_row': 'A', 'plate_column': 1, 'chemical_name': 'Perfluorodecalin', 'class': 'pfd',      'x': 0,  'y': 0,  'smiles': ''},
    {'plate_row': 'A', 'plate_column': 2, 'chemical_name': 'Amine 1',          'class': 'amine',    'x': 35, 'y': 0,  'smiles': ''},
    {'plate_row': 'A', 'plate_column': 3, 'chemical_name': 'Amine 2',          'class': 'amine',    'x': 70, 'y': 0,  'smiles': ''},
    {'plate_row': 'B', 'plate_column': 1, 'chemical_name': 'Aldehyde 1',       'class': 'aldehyde', 'x': 0,  'y': 35, 'smiles': ''},
    {'plate_row': 'B', 'plate_column': 2, 'chemical_name': 'Aldehyde 2',       'class': 'aldehyde', 'x': 35, 'y': 35, 'smiles': ''},
    {'plate_row': 'B', 'plate_column': 3, 'chemical_name': 'Wash',             'class': 'wash',     'x': 70, 'y': 35, 'smiles': ''},
])

screen = Screen(dataframe=demo_df, PFD_dwell=3.2, organic_dwell=0.4,
                 pause_after_pump_stop=0.5, repeats=3, feedrate=2000)
total_droplets = screen.initialize()
gcode_lines = screen.generate_gcode()

print(f"Reagent categories: {screen.clas}")
print(f"Total droplets: {total_droplets}  |  G-code lines: {len(gcode_lines)}")
print('\n'.join(gcode_lines[:7]), '\n...')

In [ ]:
import pandas as pd

sys.path.insert(0, APP_DIR)
from gcode_generator import Screen

demo_df = pd.DataFrame([
    {'plate_row': 'A', 'plate_column': 1, 'chemical_name': 'Perfluorodecalin', 'class': 'pfd',      'x': 0,  'y': 0,  'smiles': ''},
    {'plate_row': 'A', 'plate_column': 2, 'chemical_name': 'Amine 1',          'class': 'amine',    'x': 35, 'y': 0,  'smiles': ''},
    {'plate_row': 'A', 'plate_column': 3, 'chemical_name': 'Amine 2',          'class': 'amine',    'x': 70, 'y': 0,  'smiles': ''},
    {'plate_row': 'B', 'plate_column': 1, 'chemical_name': 'Aldehyde 1',       'class': 'aldehyde', 'x': 0,  'y': 35, 'smiles': ''},
    {'plate_row': 'B', 'plate_column': 2, 'chemical_name': 'Aldehyde 2',       'class': 'aldehyde', 'x': 35, 'y': 35, 'smiles': ''},
    {'plate_row': 'B', 'plate_column': 3, 'chemical_name': 'Wash',             'class': 'wash',     'x': 70, 'y': 35, 'smiles': ''},
])

screen = Screen(dataframe=demo_df, PFD_dwell=3.2, organic_dwell=0.4,
                 pause_after_pump_stop=0.5, repeats=3, feedrate=2000)
total_droplets = screen.initialize()
gcode_lines = screen.generate_gcode()

print(f"Reagent categories: {screen.clas}")
print(f"Total droplets: {total_droplets}  |  G-code lines: {len(gcode_lines)}")
print('\n'.join(gcode_lines[:7]), '\n...')

In [ ]:
from time_estimate import format_estimate

print(format_estimate(
    gcode_lines, feedrate_mmpm=2000,
    max_rates_mmpm=(2000.0, 500.0, 500.0),   # MACHINE SETTINGS: X/Y/Z Max Rate
    accels_mmss=(10.0, 10.0, 10.0),          # MACHINE SETTINGS: X/Y/Z Acceleration
))
screen.get_combinations_df().head()

## 5. Connect the CNC & pump

**Run tab → HARDWARE**: type the **Port** (e.g. `COM3`) and **Baud** for each device, then `Connect` (or `Connect All`). No auto-detect dropdown — find your port first:

1. Plug in the device, open **Device Manager** (right-click Start → Device Manager)
2. Expand **Ports (COM & LPT)** — e.g. `USB-SERIAL CH340 (COM3)`; the number in parentheses is your port
3. Unsure which device is which? Unplug one, hit **Action → Scan for hardware changes**, see which entry disappears

CNC baud is typically `115200` (GRBL); pump baud depends on your pump — check its manual. Whatever you enter is saved automatically.

<img src="tutorial_screenshots/run_tab_hardware.png" width="700">

Or list ports straight from Python (with hardware plugged in):

In [ ]:
from serial.tools import list_ports

for p in list_ports.comports():
    print(f"{p.device:<8} {p.description}")

## 6. Run the reaction

**Run tab → CONTROLS**: `Zero CNC (G92)` first (with the gantry manually positioned at your reference point), then **▶ Start** / **⏸ Pause** (↔ **▶ Resume**) / **⏹ Stop**.

**PROGRESS** shows live status:
- Droplet counter: `"{droplet #} / {total}  ({percent}%)"`, advancing once per carrier-fluid dip
- Time estimate (computed before the run starts, from dwell times + Machine Settings): `"Estimated run time: ~{total}  (dwell {dwell} + movement {move})"`
- A random `.gif` from `loading_images/` plays during the run — drop any animated `.gif` in that folder to add it to the rotation, no code changes needed

<img src="tutorial_screenshots/run_tab_progress.png" width="700">

## 7. Troubleshooting

| Symptom | Fix |
|---|---|
| "Install matplotlib to use this feature" | `%pip install matplotlib`, restart the app |
| "No PFD/carrier fluid well found in CSV." | Assign at least one well class `pfd` / `perfluorodecalin` / `oil` |
| "CSV missing columns: …" | CSV needs `plate_row`, `plate_column`, `chemical_name`, `class` |
| Connect fails with a port error | Wrong COM number, or the port is already open elsewhere (Arduino IDE, another instance…) |
| Pump connects but misbehaves | Baud mismatch — check the pump's manual and update the Baud field |
| No loading-screen animation | `loading_images/` is empty or missing next to `main.py` |
| Time estimate looks off | MACHINE SETTINGS ($110–$122) don't match your controller's real values |
| ▶ Start stays greyed out | No G-code generated or loaded yet (§4) |